In [4]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"

from pathlib import Path

import torch

from satchange.data import prepare_levir_cd
from satchange.experiments import (
    load_fc_siam_diff_experiment,
)
from satchange.optimization import (
    run_parameter_search,
)


current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory


EXPERIMENT = "full"

config, config_path = load_fc_siam_diff_experiment(
    experiment=EXPERIMENT,
    project_root=project_root,
)

dataset_root = prepare_levir_cd(
    config=config,
    project_root=project_root,
)


print("Experiment:", EXPERIMENT)
print("Config:", config_path)
print("Image size:", config["data"]["image_size"])
print("Visible GPUs:", torch.cuda.device_count())

assert config["data"]["image_size"] is None

Download directory is not empty: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/downloads/levir_cd
Skipping download.
Split is already extracted: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/LEVIR-CD/train
Skipping extraction.
Split is already extracted: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/LEVIR-CD/val
Skipping extraction.
Split is already extracted: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/LEVIR-CD/test
Skipping extraction.

Split:         train
Before images: 445
After images:  445
Labels:        445

Split:         val
Before images: 64
After images:  64
Labels:        64

Split:         test
Before images: 128
After images:  128
Labels:        128

Total samples: 637
LEVIR-CD structure is valid.
Experiment: full
Config: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/configs/fc_siam_diff_full.yaml
Image size: None
Visible

In [2]:
THRESHOLDS = [
    round(value / 100, 2)
    for value in range(5, 96, 5)
]


SEARCH_SPACE = {
    "learning_rate": [
        3e-4,
        6e-4,
        1e-3,
        2e-3,
    ],
}


FIXED_PARAMETERS = {
    "scheduler": "none",

    "positive_weight": 10,
    "bce_weight": 0.3,
    "dice_weight": 0.7,

    # Temporary until resized Stage 3 finishes.
    "weight_decay": 1e-4,

    # Start conservatively at full resolution.
    "batch_size": 8,

    # None = original 1024 × 1024.
    "image_size": (
        config["data"]["image_size"]
    ),

    "num_workers": 0,

    "seed": (
        config["project"]["seed"]
    ),

    "model_parameters": dict(
        config["model"]["parameters"]
    ),
}

In [3]:
full_results = run_parameter_search(
    search_space=SEARCH_SPACE,
    fixed_parameters=FIXED_PARAMETERS,
    model_name="fc_siam_diff",
    dataset_root=dataset_root,
    thresholds=THRESHOLDS,
    number_of_epochs=20,
    output_root=(
        project_root
        / "artifacts"
        / "hyperparameter_search"
    ),
    search_name="full_r1_learning_rate_screen",
    gpu_ids=[
        0,
        1,
        2,
        3,
    ],
    checkpoint_interval=5,
)

Trials: 4
GPUs: [0, 1, 2, 3]

[Trial 002 | GPU 2] Epoch 1/20
Train loss:       0.921626
Validation loss:  0.909750
Best threshold:   0.05
Precision:        0.041966
Recall:           1.000000
F1:               0.080551
IoU:              0.041966

[Trial 003 | GPU 3] Epoch 1/20
Train loss:       0.918365
Validation loss:  0.909440
Best threshold:   0.05
Precision:        0.041966
Recall:           1.000000
F1:               0.080551
IoU:              0.041966

[Trial 000 | GPU 0] Epoch 1/20
Train loss:       0.921769
Validation loss:  0.908727
Best threshold:   0.05
Precision:        0.041966
Recall:           1.000000
F1:               0.080551
IoU:              0.041966

[Trial 001 | GPU 1] Epoch 1/20
Train loss:       0.920171
Validation loss:  0.906899
Best threshold:   0.35
Precision:        0.181765
Recall:           0.073593
F1:               0.104768
IoU:              0.055280

[Trial 002 | GPU 2] Epoch 2/20
Train loss:       0.913904
Validation loss:  0.896488
Best threshold:  